# Milestone 5 lab, notebook 5 of 7: The days you missed

## Where we are

| Notebook | What you do |
|---|---|
| 1. A clock is not enough | Start two steps by the clock, and get yesterday's report |
| 2. Tasks and arrows | Read the DAG, and run it once |
| 3. Which day is this run for? | The logical date, the data interval, and a run you trigger by hand |
| 4. When a task fails | Break the API, then break the data, and watch what Airflow does |
| **5. The days you missed** (you are here) | Run a week with one command. Run one day again |
| 6. Every five minutes | A lab: the same pipeline as a micro-batch |
| 7. The bad morning | The drill: the API is down, then a page comes twice |

Laptops sleep. Servers are restarted. Sometimes a pipeline does not run for a week. This notebook is about the days that were missed, and about doing a day again.

In [1]:
from lab_helpers import MILESTONE_DIR, at, report_is_right, schedule_warehouse, show_doc, show_runs, start_fresh, table

at.wait_until_ready()
start_fresh()

Airflow has no runs. The warehouse is empty. The order system is back at its first morning.


## Step 1: What should a scheduler do about missed days?

The DAG's first business day is 1 June 2026. On your computer's real calendar that is in the past. So, as far as Airflow can tell, this DAG has **missed every one of its runs**.

A scheduler can react in two ways. The setting is called **catchup**.

In [2]:
dag_file = (MILESTONE_DIR / "dags" / "daily_orders_dag.py").read_text()
print(dag_file[dag_file.index("    catchup="):dag_file.index("    default_args")])

    catchup=False,  # do not run every missed morning by itself. A person asks for a backfill
    max_active_runs=1,  # one morning at a time. Two runs must never write the same tables together



| `catchup` | When the scheduler finds missed days | 
|---|---|
| `True` | It starts a run for **every** missed day, by itself, at once |
| `False` | It starts nothing old. A person asks for the days they want |

This DAG says `False`, and the empty list below is the proof: Airflow is running, the DAG is switched on, and it has started nothing.

In [3]:
print("Runs so far:", at.runs())

Runs so far: []


`False` is the careful choice. After a week's outage, `True` would throw seven runs at the order system the moment the scheduler wakes up. With `False`, a person decides when that happens.

## Step 2: A backfill

Running the job for days in the past is called a **backfill**. Here is one command for five business days:

```
airflow backfill create --dag-id daily_orders \
    --from-date 2026-06-07 --to-date 2026-06-12 --max-active-runs 1
```

`at.backfill` runs it and waits.

In [4]:
at.backfill("2026-06-07", "2026-06-11")
show_runs()

,business_day,started_by,state
0,2026-06-07,backfill,success
1,2026-06-08,backfill,success
2,2026-06-09,backfill,success
3,2026-06-10,backfill,success
4,2026-06-11,backfill,success


Five runs, each started by `backfill`, each for its own business day. Did they run in order, and one at a time?

In [5]:
con = schedule_warehouse()
table(con, "SELECT run_id, run_date AS morning, asked_since, rows_landed, tests_passed FROM ops.load_runs ORDER BY loaded_at")

,run_id,morning,asked_since,rows_landed,tests_passed
0,2026-06-08_1,2026-06-08,NaT,226,31
1,2026-06-09_1,2026-06-09,2026-06-06 03:48:53+00:00,102,31
2,2026-06-10_1,2026-06-10,2026-06-07 04:17:31+00:00,93,31
3,2026-06-11_1,2026-06-11,2026-06-08 04:11:39+00:00,120,31
4,2026-06-12_1,2026-06-12,2026-06-09 04:20:36+00:00,103,31


Oldest first, one after the other. Each run's `asked_since` builds on the watermark the run before it left. That is why `--max-active-runs 1` is in the command: two mornings must never write the same tables at the same moment.

In [6]:
print("The report equals the answer key for the morning of 12 June:", report_is_right("2026-06-12"))

The report equals the answer key for the morning of 12 June: True


One command did five mornings, and the report is right.

A backfill is not a special kind of run. Each of the five did exactly what a scheduled run would have done. It works for the same two reasons as everything in this milestone: every run takes its day from its **logical date**, and every step is **idempotent**.

## Step 3: Do one day again

Finance asks: "Can you run last Tuesday again? We want to be sure." Running a finished day again is a **replay**. In Airflow you **clear** the run: its tasks lose their "done" mark, and the scheduler runs them again.

```
airflow tasks clear daily_orders -s 2026-06-09 -e 2026-06-10 -y
```

First, take a copy of everything people read.

In [7]:
def snapshot():
    orders = con.execute("SELECT order_id, status, total_cents, refunded_cents, is_deleted FROM core.orders ORDER BY 1").fetchall()
    report = con.execute("SELECT * FROM marts.daily_orders ORDER BY 1, 2").fetchall()
    return orders, report

before = snapshot()
at.replay("2026-06-09")
after = snapshot()

print("Orders identical after the replay:", before[0] == after[0], f"({len(after[0])} rows)")
print("Report identical after the replay:", before[1] == after[1], f"({len(after[1])} rows)")

Orders identical after the replay: True (347 rows)
Report identical after the replay: True (33 rows)


The day ran again, and nothing changed. The log shows that it really did run:

In [8]:
table(con, "SELECT run_id, run_date AS morning, rows_landed, tests_passed FROM ops.load_runs ORDER BY loaded_at")

,run_id,morning,rows_landed,tests_passed
0,2026-06-08_1,2026-06-08,226,31
1,2026-06-09_1,2026-06-09,102,31
2,2026-06-10_1,2026-06-10,93,31
3,2026-06-11_1,2026-06-11,120,31
4,2026-06-12_1,2026-06-12,103,31
5,2026-06-10_2,2026-06-10,0,31


The last row is the replay: a second run for the morning of 10 June.

## Step 4: Three words that are easy to mix up

| Word | It means | Who starts it | Here |
|---|---|---|---|
| **Catchup** | The scheduler running missed days **by itself** | The scheduler | Switched off |
| **Backfill** | Running the job for a range of past days | A person | Five days, one command |
| **Replay** | Running one finished day again | A person | Tuesday 9 June, again |

A retry is something else again: a retry belongs to one **task** that failed inside a run. A replay is a whole run, done again on purpose.

### A safe backfill, in six steps

For a real backfill of many days, a team follows a plan like this:

1. Oldest day first.
2. One day at a time.
3. Overwrite each day. Never add to it.
4. Compare the count and the total with the source.
5. Stop at the first mismatch.
6. Write down which days are finished.

This pipeline does the first three by its design. Step 4 is the table test "the report equals its orders".

In [9]:
con.close()
show_doc("HLD.md", "5. Three ways a run begins")

**From `docs/HLD.md`**

> ## 5. Three ways a run begins
> 
> | Way | Who starts it | Used for |
> |---|---|---|
> | **Schedule** | Airflow, when a data interval ends | Every normal morning |
> | **Trigger** | A person, for one day | A day that has no run yet |
> | **Backfill** | A person, for a range of days | Days that were missed |
> 
> And one way to do a day **again**: a **replay**. A person clears a finished run, and Airflow runs its tasks again.
> 
> What about days the scheduler missed while it was switched off? That is the **catchup** setting.
> 
> | `catchup` | When the scheduler comes back after 5 days off | Chosen here |
> |---|---|---|
> | `True` | It starts all 5 missed runs by itself | No |
> | `False` | It starts nothing old. A person asks for a backfill | **Yes** |
> 
> With `False`, a person decides when a pile of old runs hits the order system.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Catchup** | The setting that makes a scheduler run the days it missed, by itself | `catchup=False` |
| **Backfill** | "A backfill runs the job again for days in the past." | Five business days, one command |
| **Replay** | Running a finished day again. In Airflow: clear the run | Tuesday 9 June |

The definitions follow the D8Loop chapters [Orchestration & Scheduling](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/orchestration-and-scheduling) and [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills).

## What you now know

- With `catchup=False` the scheduler does not run missed days by itself.
- A backfill runs a range of past days with one command, oldest first, one at a time.
- A replay runs a finished day again, and changes nothing.
- Both work because a run takes its day from its logical date, and every step is idempotent.

**Say it in an interview:** "I keep catchup off, so after an outage I decide when the old days run. Then I backfill the range with max-active-runs set to one, oldest first. I also replay single days by clearing the run. I tested both: a five-day backfill reproduced the answer key, and a replay left every row of the orders table and the report unchanged."

Next: [notebook 6](06_every_five_minutes.ipynb) pushes the batch as far as it goes.